# core

> Using pyinstrument to profile FastHTML apps

Sometimes when building FastHTML apps we run into performance bottlenecks. Figuring out what is slow can be challenging, especially when building apps with async components. That's where profiling tools like pyinstrument can help. Profilers are tools that show exactly how long each component of a project takes to run. Identifying slow parts of an app is the first step in figuring out how to make things run faster.

In [ ]:
#| default_exp core

In [ ]:
#| export
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware import Middleware
try: from rich import print
except: pass
try:
    from pyinstrument import Profiler
except ImportError:
    raise ImportError('Please install pyinstrument')

In [ ]:
#| export
class ProfileMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        profiling = request.query_params.get("profile", False)
        terminal = request.query_params.get("terminal", False)  
        html = request.query_params.get('write', False)
        if profiling:
            profiler = Profiler()
            profiler.start()
            response = await call_next(request)            
            profiler.stop()
            if terminal: print(profiler.output_text())
            if html: profiler.write_html('profile.html')
            return HTMLResponse(profiler.output_html())
        return await call_next(request)

# app, rt = fast_app(middleware=(Middleware(ProfileMiddleware)))

## Tests

In [ ]:
from starlette.testclient import TestClient
from fastcore.all import *
from httpx import ASGITransport, AsyncClient
from functools import partialmethod
from anyio import from_thread

In [ ]:
class Client:
    "A simple httpx ASGI client that doesn't require `async`"
    def __init__(self, app, url="http://testserver"):
        self.cli = AsyncClient(transport=ASGITransport(app), base_url=url)

    def _sync(self, method, url, **kwargs):
        async def _request(): return await self.cli.request(method, url, **kwargs)
        with from_thread.start_blocking_portal() as portal: return portal.call(_request)

for o in ('get', 'post', 'delete', 'put', 'patch', 'options'): setattr(Client, o, partialmethod(Client._sync, o))

In [ ]:
app, rt = fast_app(middleware=(Middleware(ProfileMiddleware)))
client = Client(app)

First, confirm that the view works normally

In [ ]:
@rt
def index(): return Titled('Hello, profiler')
'Hello, profiler' in client.get('/').text

True

Now lets profile it! Or rather, check that it works.

In [ ]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

Let's print to the terminal

In [ ]:
client.get('/?profile=1&terminal=1')

_     ._   __/__   _ _  _  _ _/_   Recorded: 08:10:06  Samples:  3
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.004     CPU time: 0.005
/   _/                      v5.0.1

Profile at /var/folders/3k/p8fttyg52s30bj9vdr1v7c180000gn/T/ipykernel_49163/2684603318.py:9

0.003 Handle._run  asyncio/events.py:86
`- 0.003 coro  starlette/middleware/base.py:135
      [28 frames hidden]  starlette, fasthtml, copy, fastcore, ...
         0.001 _reconstruct  copy.py:247
         0.001 str.lower  <built-in>
         0.001 WeakKeyDictionary.__getitem__  weakref.py:414

<Response [200 OK]>

In [ ]:
client.get('/?profile=1&write=1')

<Response [200 OK]>

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()